# 5. Figures 1–3 of the paper

No GPU needed (a CPU runtime is enough). Run everything once; afterwards change DPI, sizes, fonts or colors in the **settings cell (2)** and re-run only the figure cell you need.

- Figure 1: clustering (t-SNE, per-paper silhouette boxplots and the difference from BGE). t-SNE coordinates are computed once and cached on Drive, so later style changes are fast.
- Figure 2: AUROC by negative type and the ablation of explicit negatives (values copied from the evaluation reports into the cell)
- Figure 3: external validation (Reactome, MeSH), per-query AP boxplots

Figures 1 and 3 include a **paired-difference panel (KEGG-BGE − BGE)** and **significance marks (\*)**. They are recomputed exactly as in `evaluate.py` (per-paper differences, cluster bootstrap over pathways or descriptors, 1,000 resamples, seed 42), so the seed-42 values should equal the Δ in Tables 2 and S6. \* = 95% CI excludes zero.

Outputs are written to `BASE/figures/`.

## 1. Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/kegg-bge')     # same Drive folder as notebooks 1–4
OUT  = BASE / 'figures'
DATA = BASE / 'data/splits'          # pathways_test.json (KEGG)
EMB  = BASE / 'embeddings/test'      # KEGG test embeddings (.npy) + pmids.json
EVAL_KEGG, EVAL_REACTOME, EVAL_MESH = BASE / 'eval', BASE / 'eval_reactome', BASE / 'eval_mesh'
# pathways_test.json of each set (cluster bootstrap unit = finest label)
SETS = {'kegg':     (DATA,                       'pathway_id'),
        'reactome': (BASE / 'external/reactome', 'pathway_id'),
        'mesh':     (BASE / 'external/mesh',     'level4')}
OUT.mkdir(parents=True, exist_ok=True)

## 2. Settings (change these, then re-run the figure cells)

- `DPI`: PNG resolution; journals usually ask for 300–600 (PDF is vector; DPI only affects the rasterized scatter points)
- `FIGSIZE`: inches. Single column ≈ 3.5, full width ≈ 7.2
- `FORMATS`: file types, e.g. `('png', 'pdf', 'tiff')`

In [ ]:
DPI      = 300
FORMATS  = ('png', 'pdf')
FONT     = 'DejaVu Sans'             # e.g. 'Arial' (see the font cell below if it is not installed)
FONTSIZE = dict(base=8, title=9, label=8, tick=7.5, legend=7, panel=11)

FIGSIZE = {1: (7.2, 7.2), 2: (7.2, 2.9), 3: (7.2, 5.8)}

OURS, BGE_C = '#D55E00', '#0072B2'
# display name, model key in pooling_choice.json, color
MODELS = [
    ('TF-IDF',        'tfidf',          '#BBBBBB'),
    ('PubMedBERT',    'pubmedbert',     '#999999'),
    ('MedCPT (T+A)',  'medcpt_native',  '#CC79A7'),
    ('SPECTER (T+A)', 'specter_native', '#009E73'),
    ('BGE',           'bge',            BGE_C),
    ('KEGG-BGE',      'kegg_bge_s42',   OURS),
]

# Figure 1 (t-SNE)
N_SHOW      = 10      # number of test pathways shown in color (largest first)
PERPLEXITY  = 30
TSNE_SEED   = 0
POINT_SIZE  = dict(grey=3, colored=7)
FIG1_SIL_MODELS = ['SPECTER (T+A)', 'BGE', 'KEGG-BGE']   # models in the silhouette boxplot (c)

# Figure 3
FIG3_MODELS = [m[0] for m in MODELS]                     # remove models here to leave them out

# Paired difference (KEGG-BGE − BGE)
SEEDS   = [('seed 42', 'kegg_bge_s42', 'o'), ('seed 1', 'kegg_bge_s1', 's'), ('seed 2', 'kegg_bge_s2', '^')]
STAR_SEED = 'kegg_bge_s42'   # the * on boxplots uses this seed (the model in the paper tables)
N_BOOT, BOOT_SEED = 1000, 42
DELTA_WIDTH = 0.32           # width of the difference panel relative to the boxplot

In [ ]:
# (optional) another font, e.g. Liberation Sans (same metrics as Arial):
# !apt-get -qq install fonts-liberation > /dev/null
# import matplotlib.font_manager as fm; fm._load_fontmanager(try_read_cache=False); FONT = 'Liberation Sans'

## 3. Helpers

In [ ]:
import json
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def style():
    plt.rcParams.update({
        'font.family': FONT, 'font.size': FONTSIZE['base'], 'axes.titlesize': FONTSIZE['title'],
        'axes.labelsize': FONTSIZE['label'], 'xtick.labelsize': FONTSIZE['tick'], 'ytick.labelsize': FONTSIZE['tick'],
        'legend.fontsize': FONTSIZE['legend'], 'axes.spines.top': False, 'axes.spines.right': False,
        'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
        'figure.dpi': 110, 'savefig.dpi': DPI, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    })

def color_of(name):
    return {n: c for n, _, c in MODELS}[name]

def panel_label(ax, s, x=-0.08, y=1.04):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=FONTSIZE['panel'], fontweight='bold', va='bottom', ha='right')

def save(fig, name):
    for ext in FORMATS:
        kw = {'pil_kwargs': {'compression': 'tiff_lzw'}} if ext in ('tif', 'tiff') else {}
        fig.savefig(OUT / f'{name}.{ext}', bbox_inches='tight', dpi=DPI, **kw)
    print('saved:', ', '.join(f'{name}.{e}' for e in FORMATS), f'({DPI} dpi) ->', OUT)
    plt.show()

def load_scores(eval_dir):
    """Per-paper test scores of each model's chosen pooling variant -> {display name: DataFrame}."""
    choice = json.load(open(eval_dir / 'pooling_choice.json'))
    df = pd.read_csv(eval_dir / 'per_paper_scores.csv.gz', dtype={'pmid': str})
    df = df[df['split'] == 'test']
    out = {}
    for name, key, _ in MODELS:
        v = choice.get(key)
        if v is None or v not in set(df['variant']):
            print(f'  [{eval_dir.name}] {key}: not found, skipped')
            continue
        out[name] = df[df['variant'] == v].set_index('pmid')
    return out

def load_variants(eval_dir, keys):
    # Per-paper scores of arbitrary model keys (e.g. the three KEGG-BGE seeds) -> {key: DataFrame}.
    choice = json.load(open(eval_dir / 'pooling_choice.json'))
    df = pd.read_csv(eval_dir / 'per_paper_scores.csv.gz', dtype={'pmid': str})
    df = df[df['split'] == 'test']
    out = {}
    for key in keys:
        v = choice.get(key)
        if v is None or v not in set(df['variant']):
            print(f'  [{eval_dir.name}] {key}: not found, skipped')
            continue
        out[key] = df[df['variant'] == v].set_index('pmid')
    return out

def paper_clusters(data_dir, finest_key):
    # pmid -> cluster label (alphabetically first finest label, as in evaluate.py).
    labels = defaultdict(set)
    for e in json.load(open(Path(data_dir) / 'pathways_test.json', encoding='utf-8')):
        for r in e['references']:
            labels[str(r['pmid'])].add(str(e[finest_key]))
    return {p: sorted(s)[0] for p, s in labels.items()}

def cluster_bootstrap_ci(values, clusters):
    # Same procedure as evaluate.py: resample clusters with replacement, ratio of summed values.
    ok = ~np.isnan(values)
    v, c = values[ok], clusters[ok]
    uniq, inv = np.unique(c, return_inverse=True)
    sums, ns = np.bincount(inv, weights=v), np.bincount(inv).astype(float)
    rng = np.random.default_rng(BOOT_SEED)
    draws = rng.multinomial(len(uniq), np.full(len(uniq), 1 / len(uniq)), size=N_BOOT)
    return np.percentile((draws @ sums) / (draws @ ns), [2.5, 97.5])

def paired_delta(a, b, col, clusters):
    # Mean of (a - b) over papers with both values, and its 95% cluster-bootstrap CI.
    idx = a.index.intersection(b.index)
    d = (a.loc[idx, col] - b.loc[idx, col]).values.astype(float)
    cl = np.array([clusters.get(p, p) for p in idx])
    lo, hi = cluster_bootstrap_ci(d, cl)
    return float(np.nanmean(d)), float(lo), float(hi)

def delta_table(dset, cols):
    # Δ (each KEGG-BGE seed − BGE) for every column -> DataFrame (seed, col, mean, lo, hi, sig).
    rows = []
    for label, key, _ in SEEDS:
        if key not in SEEDV[dset]:
            continue
        for col in cols:
            m, lo, hi = paired_delta(SEEDV[dset][key], SCORES[dset]['BGE'], col, CLUSTERS[dset])
            rows.append(dict(seed=label, key=key, col=col, mean=m, lo=lo, hi=hi, sig=(lo > 0) or (hi < 0)))
    return pd.DataFrame(rows)

def delta_panel(ax, dt, cols, ticklabels, ylabel):
    # Points = Δ per seed with 95% CI; filled = significant.
    ax.axhline(0, color='#888888', lw=0.6, ls='--')
    off = np.linspace(-0.18, 0.18, len(SEEDS))
    for j, (label, key, marker) in enumerate(SEEDS):
        sub = dt[dt['key'] == key].set_index('col').reindex(cols)
        x = np.arange(len(cols)) + off[j]
        ax.errorbar(x, sub['mean'], yerr=[sub['mean'] - sub['lo'], sub['hi'] - sub['mean']], fmt='none',
                    ecolor=OURS, elinewidth=0.8, capsize=1.5)
        for xi, (_, r) in zip(x, sub.iterrows()):
            ax.scatter(xi, r['mean'], marker=marker, s=16, zorder=3, edgecolor=OURS, linewidth=0.8,
                       color=OURS if r['sig'] else 'white')
        ax.scatter([], [], marker=marker, s=16, color=OURS, label=f'KEGG-BGE {label}')
    ax.set_xticks(np.arange(len(cols)), ticklabels)
    ax.set_xlim(-0.5, len(cols) - 0.5)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', lw=0.3, color='#E5E5E5')

def star(ax, x, data, sig, pad=0.02):
    # Put * above a box (top of its upper whisker) when sig.
    if not sig:
        return
    q1, q3 = np.percentile(data, [25, 75])
    top = min(np.max(data), q3 + 1.5 * (q3 - q1))
    ax.text(x, min(top + pad, ax.get_ylim()[1] - 0.04 * np.ptp(ax.get_ylim())), '*', ha='center', va='bottom', fontsize=FONTSIZE['panel'], fontweight='bold', color=OURS)

def box_x(j, n, pos, width):
    return pos + (j - (n - 1) / 2) * width

def boxes(ax, groups, names, positions, width):
    """One box per (group, model); no fliers; white diamond = mean."""
    for j, name in enumerate(names):
        data = [g[j] for g in groups]
        pos = [p + (j - (len(names) - 1) / 2) * width for p in positions]
        bp = ax.boxplot(data, positions=pos, widths=width * 0.8, patch_artist=True, showfliers=False,
                        medianprops=dict(color='black', lw=0.8), whiskerprops=dict(lw=0.6),
                        capprops=dict(lw=0.6), boxprops=dict(lw=0.6))
        for b in bp['boxes']:
            b.set_facecolor(color_of(name)); b.set_alpha(0.9)
        ax.scatter(pos, [np.mean(d) for d in data], marker='D', s=9, color='white', edgecolor='black',
                   linewidth=0.5, zorder=3)

def legend_handles(names):
    return [Patch(facecolor=color_of(n), edgecolor='black', linewidth=0.5, label=n) for n in names]

EVALS    = {'kegg': EVAL_KEGG, 'reactome': EVAL_REACTOME, 'mesh': EVAL_MESH}
SCORES   = {d: load_scores(e) for d, e in EVALS.items()}
SEEDV    = {d: load_variants(e, [k for _, k, _ in SEEDS]) for d, e in EVALS.items()}
CLUSTERS = {d: paper_clusters(*SETS[d]) for d in EVALS}
print({d: len(set(c.values())) for d, c in CLUSTERS.items()}, 'clusters per set')

## Figure 1. Clustering

### 1-1. t-SNE (first run only, a few minutes)
Coordinates are cached in `figures/tsne_cache.npz`; changing `PERPLEXITY` or `TSNE_SEED` triggers a new computation.

In [ ]:
from sklearn.manifold import TSNE

pathways = json.load(open(DATA / 'pathways_test.json', encoding='utf-8'))
pmids = json.load(open(EMB / 'pmids.json'))
choice = json.load(open(EVAL_KEGG / 'pooling_choice.json'))
TSNE_MODELS = [('BGE', 'bge'), ('KEGG-BGE', 'kegg_bge_s42')]

cache = OUT / 'tsne_cache.npz'
tag = f'p{PERPLEXITY}_s{TSNE_SEED}'
TSNE_XY = dict(np.load(cache)) if cache.exists() else {}
for name, key in TSNE_MODELS:
    k = f'{key}_{tag}'
    if k not in TSNE_XY:
        E = np.load(EMB / f'{choice[key]}.npy')
        E = E / np.linalg.norm(E, axis=1, keepdims=True)
        print('t-SNE:', name, E.shape)
        TSNE_XY[k] = TSNE(n_components=2, metric='cosine', init='pca', perplexity=PERPLEXITY,
                          random_state=TSNE_SEED).fit_transform(E)
np.savez(cache, **TSNE_XY)
print('done')

### 1-2. Plot

In [ ]:
style()
member, pw_name = defaultdict(set), {}
for e in pathways:
    pw_name[e['pathway_id']] = e.get('pathway_name', e['pathway_id'])
    for r in e['references']:
        member[r['pmid']].add(e['pathway_id'])
single = Counter(next(iter(s)) for s in member.values() if len(s) == 1)
shown = [pw for pw, _ in single.most_common(N_SHOW)]
cmap = plt.get_cmap('tab10' if N_SHOW <= 10 else 'tab20')
pw_color = {pw: cmap(i) for i, pw in enumerate(shown)}
lab = np.array([next(iter(member[p])) if len(member[p]) == 1 and next(iter(member[p])) in pw_color else ''
                for p in pmids])

fig = plt.figure(figsize=FIGSIZE[1])
gs = fig.add_gridspec(3, 2, height_ratios=[1.15, 0.24, 1], hspace=0.12, wspace=0.12)
for k, (name, key) in enumerate(TSNE_MODELS):
    Y = TSNE_XY[f'{key}_{tag}']
    ax = fig.add_subplot(gs[0, k])
    g = lab == ''
    ax.scatter(Y[g, 0], Y[g, 1], s=POINT_SIZE['grey'], color='#DDDDDD', linewidth=0, rasterized=True)
    for pw in shown:
        m = lab == pw
        ax.scatter(Y[m, 0], Y[m, 1], s=POINT_SIZE['colored'], color=pw_color[pw], linewidth=0,
                   label=pw_name[pw], rasterized=True)
    sil = np.nanmean(SCORES['kegg'][name]['pathway_silhouette'])
    ax.set_title(f'{name}  (pathway silhouette {sil:.3f})', color=color_of(name))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.5); s.set_color('#888888')
    panel_label(ax, 'ab'[k], x=-0.02)
h, l = ax.get_legend_handles_labels()
lax = fig.add_subplot(gs[1, :]); lax.axis('off')
lax.legend(h, [t if len(t) < 40 else t[:38] + '…' for t in l], loc='center', ncol=3, frameon=False,
           markerscale=2, handletextpad=0.2, columnspacing=1)

sub = gs[2, :].subgridspec(1, 2, width_ratios=[1, DELTA_WIDTH], wspace=0.28)
ax = fig.add_subplot(sub[0])
sets = [('KEGG\n(pathway)', 'kegg', 'pathway_silhouette'),
        ('Reactome\n(pathway)', 'reactome', 'pathway_silhouette'),
        ('MeSH\n(Level 4)', 'mesh', 'level4_silhouette')]
groups = [[SCORES[d][n][col].dropna().values for n in FIG1_SIL_MODELS] for _, d, col in sets]
boxes(ax, groups, FIG1_SIL_MODELS, positions=np.arange(len(sets)), width=0.22)
ax.axhline(0, color='#888888', lw=0.5, ls=':')
ax.set_xticks(np.arange(len(sets)), [s[0] for s in sets])
ax.set_ylabel('Per-paper silhouette (cosine)')
panel_label(ax, 'c', x=-0.06)

# * above KEGG-BGE boxes + (d) paired difference panel
fig1_delta = pd.concat([delta_table(d, [col]).assign(set=d) for _, d, col in sets], ignore_index=True)
fig1_delta['col'] = fig1_delta['set']            # one column per data set in the panel
j = FIG1_SIL_MODELS.index('KEGG-BGE') if 'KEGG-BGE' in FIG1_SIL_MODELS else None
if j is not None:
    for i, (_, d, col) in enumerate(sets):
        r = fig1_delta[(fig1_delta['set'] == d) & (fig1_delta['key'] == STAR_SEED)]
        star(ax, box_x(j, len(FIG1_SIL_MODELS), i, 0.22), groups[i][j], bool(r['sig'].iloc[0]) if len(r) else False)
dax = fig.add_subplot(sub[1])
delta_panel(dax, fig1_delta, [d for _, d, _ in sets], ['KEGG', 'React.', 'MeSH'], 'Δ silhouette vs BGE')
panel_label(dax, 'd', x=-0.3)
h, l = dax.get_legend_handles_labels()
fig.legend(legend_handles(FIG1_SIL_MODELS) + h, FIG1_SIL_MODELS + l, loc='upper center', bbox_to_anchor=(0.5, 0.045),
           ncol=6, frameon=False, handlelength=1.2, columnspacing=1, handletextpad=0.3)
save(fig, 'figure1_clustering')

# values to check against the text and tables
for _, d, col in sets:
    print(d, {n: round(float(np.nanmean(SCORES[d][n][col])), 3) for n in FIG1_SIL_MODELS})
print(fig1_delta[['set', 'seed', 'mean', 'lo', 'hi', 'sig']].round(3).to_string(index=False))

## Figure 2. AUROC by negative type and ablation

Values are copied from the evaluation reports (KEGG test, seed 42).

In [ ]:
style()
NEG = {  # AUROC, positives vs hard / standard / soft
    'TF-IDF':         ([0.803, 0.897, 0.946], '#BBBBBB', 'o', '-'),
    'SPECTER2 (T+A)': ([0.841, 0.928, 0.959], '#009E73', 'o', '-'),
    'Cross-encoder':  ([0.829, 0.906, 0.931], '#7F7F7F', 's', '--'),
    'BGE':            ([0.852, 0.939, 0.973], BGE_C, 'o', '-'),
    'KEGG-BGE':       ([0.893, 0.966, 0.986], OURS, 'o', '-'),
}
ABL = [  # explicit negatives: (pathway mAP, AUROC vs hard); lr 5e-6, seed 42
    ('In-batch only',    0.766, 0.892),
    ('Hard only',        0.763, 0.894),
    ('Soft only',        0.768, 0.892),
    ('All three (main)', 0.764, 0.893),
]
ABL_SEED = {'map': (0.762, 0.002), 'hard': (0.892, 0.001)}   # main model mean, SD over 3 seeds
ABL_BGE  = {'map': 0.718, 'hard': 0.852}

fig = plt.figure(figsize=FIGSIZE[2])
gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1, 1], wspace=0.55)
ax = fig.add_subplot(gs[0])
x = np.arange(3)
for name, (vals, c, m, ls) in NEG.items():
    ax.plot(x, vals, ls, marker=m, color=c, lw=1.2 if name == 'KEGG-BGE' else 0.9, ms=4, label=name)
ax.set_xticks(x, ['Hard', 'Standard', 'Soft'])
ax.set_xlabel('Negative type'); ax.set_xlim(-0.3, 2.3); ax.set_ylim(0.76, 1.0)
ax.set_ylabel('AUROC, positives vs negatives')
ax.legend(frameon=False, loc='lower right', handlelength=1.6, borderaxespad=0.2)
ax.grid(axis='y', lw=0.3, color='#E5E5E5')
panel_label(ax, 'a', x=-0.18)

for k, (key, xlabel) in enumerate([('map', 'Pathway mAP'), ('hard', 'AUROC vs hard negatives')]):
    ax = fig.add_subplot(gs[k + 1])
    y = np.arange(len(ABL))[::-1]
    vals = [a[1 + k] for a in ABL]
    mu, sd = ABL_SEED[key]
    ax.axvspan(mu - sd, mu + sd, color=OURS, alpha=0.15, lw=0)
    ax.axvline(ABL_BGE[key], color=BGE_C, ls='--', lw=0.9)
    ax.scatter(vals, y, color=OURS, s=22, zorder=3)
    ax.text(ABL_BGE[key], len(ABL) - 0.45, ' BGE', color=BGE_C, fontsize=FONTSIZE['legend'], va='center')
    ax.set_xlim(ABL_BGE[key] - 0.01, max(vals) + 0.012)
    ax.set_ylim(-0.6, len(ABL) - 0.2)
    ax.set_yticks(y, [a[0] for a in ABL] if k == 0 else [])
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', lw=0.3, color='#E5E5E5')
    if k == 0:
        panel_label(ax, 'b', x=-0.05)
fig.text(0.98, -0.04, 'Shaded band: main model mean ± SD over three seeds', ha='right',
         fontsize=FONTSIZE['legend'] - 0.5, color='#555555')
save(fig, 'figure2_negatives')

## Figure 3. External validation (Reactome, MeSH)

In [ ]:
style()
sets = [('Reactome', 'reactome', [('category', 'Category'), ('subcategory', 'Subcategory'), ('pathway', 'Pathway')]),
        ('MeSH', 'mesh', [('level1', 'Level 1'), ('level2', 'Level 2'), ('level3', 'Level 3'), ('level4', 'Level 4')])]
fig = plt.figure(figsize=FIGSIZE[3])
gs = fig.add_gridspec(2, 2, width_ratios=[1, DELTA_WIDTH], hspace=0.42, wspace=0.25)
fig3_delta = []
for k, (title, d, levels) in enumerate(sets):
    ax = fig.add_subplot(gs[k, 0])
    sc = SCORES[d]
    names = [n for n in FIG3_MODELS if n in sc]
    groups = [[sc[n][f'{lv}_map'].dropna().values for n in names] for lv, _ in levels]
    w = 0.8 / len(names)
    boxes(ax, groups, names, positions=np.arange(len(levels)), width=w)
    cols = [f'{lv}_map' for lv, _ in levels]
    dt = delta_table(d, cols)
    fig3_delta.append(dt.assign(set=d))
    if 'KEGG-BGE' in names:
        j = names.index('KEGG-BGE')
        for i, col in enumerate(cols):
            r = dt[(dt['col'] == col) & (dt['key'] == STAR_SEED)]
            star(ax, box_x(j, len(names), i, w), groups[i][j], bool(r['sig'].iloc[0]) if len(r) else False)
    ax.set_xticks(np.arange(len(levels)), [l for _, l in levels])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Average precision per query')
    ax.set_title(title, loc='left', fontweight='bold')
    ax.grid(axis='y', lw=0.3, color='#E5E5E5')
    panel_label(ax, 'ab'[k], x=-0.06)
    if k == 0:
        ax.legend(handles=legend_handles(names), ncol=len(names), frameon=False, loc='lower left',
                  bbox_to_anchor=(0, 1.08), handlelength=1.2, columnspacing=1)
    dax = fig.add_subplot(gs[k, 1])
    delta_panel(dax, dt, cols, [l.replace('Subcategory', 'Subcat.').replace('Category', 'Cat.').replace('Level ', 'L')
                                for _, l in levels], 'Δ AP vs BGE')
    dax.tick_params(axis='x', labelsize=FONTSIZE['tick'] - 0.5)
    panel_label(dax, 'cd'[k], x=-0.3)
h, l = dax.get_legend_handles_labels()
fig.legend(h, l, loc='upper right', bbox_to_anchor=(0.98, 0.05), ncol=3, frameon=False, handletextpad=0.2,
           columnspacing=1, title='Panels c, d: filled = 95% CI excludes 0', title_fontsize=FONTSIZE['legend'])
save(fig, 'figure3_external')
fig3_delta = pd.concat(fig3_delta, ignore_index=True)
print(fig3_delta[['set', 'col', 'seed', 'mean', 'lo', 'hi', 'sig']].round(3).to_string(index=False))

# values to check against the text (= mAP)
for _, d, levels in sets:
    print(d, pd.DataFrame({n: [SCORES[d][n][f'{lv}_map'].mean() for lv, _ in levels] for n in FIG3_MODELS if n in SCORES[d]},
                          index=[l for _, l in levels]).T.round(3).to_string(), sep='\n')